# W&B Leaderboard --- Tamper Detection Ablation Study

Centralized experiment analysis notebook that retrieves results from Weights & Biases
and builds:

- **Experiment leaderboard** (Pixel F1, Image Accuracy)
- **Evaluation tables** (feature set comparison, full metrics)
- **Performance comparison graphs** (bar charts, ablation curves)
- **Training curve visualization** (loss, F1, LR per epoch)
- **Prediction examples** (original / GT / predicted mask)
- **Summary statistics** for research-style reporting

---

## Step 1 --- Authenticate with W&B

In [ ]:
!pip install -q wandb pandas tabulate matplotlib seaborn

import os
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from tabulate import tabulate
from IPython.display import display, HTML

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# --- Authenticate ---
WANDB_API_KEY = os.environ.get('WANDB_API_KEY')
WANDB_USERNAME = os.environ.get('WAND_USERNAME')  # Kaggle secret name

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
else:
    wandb.login()

print('W&B authenticated successfully.')

## Step 2 --- Connect to Project and Fetch Runs

In [ ]:
api = wandb.Api()

WANDB_PROJECT = 'Tampered Image Detection & Localization'
entity = WANDB_USERNAME or api.default_entity

runs = api.runs(f'{entity}/{WANDB_PROJECT}')
print(f'Project: {entity}/{WANDB_PROJECT}')
print(f'Found {len(runs)} total runs')

## Step 3 --- Extract Metrics into DataFrame

In [ ]:
records = []
for run in runs:
    cfg = run.config
    s = run.summary._json_dict

    records.append({
        # --- identifiers ---
        'experiment': cfg.get('experiment', run.name or ''),
        'version': cfg.get('version', run.name or ''),
        'change': cfg.get('change', ''),
        'run_tag': cfg.get('run', ''),
        'wandb_run_id': run.id,
        'state': run.state,
        # --- config ---
        'feature_set': cfg.get('feature_set', ''),
        'encoder': cfg.get('encoder', ''),
        'input_type': cfg.get('input_type', cfg.get('feature_set', '')),
        'in_channels': cfg.get('in_channels', ''),
        'epochs': cfg.get('epochs', ''),
        'learning_rate': cfg.get('learning_rate', ''),
        'batch_size': cfg.get('batch_size', ''),
        'img_size': cfg.get('img_size', ''),
        'tta': cfg.get('tta', False),
        'jpeg_aug': cfg.get('jpeg_aug', False),
        'edge_supervision': cfg.get('edge_supervision', False),
        'noise_features': cfg.get('noise_features', False),
        # --- pixel-level metrics ---
        'pixel_f1': s.get('pixel_f1'),
        'pixel_iou': s.get('pixel_iou'),
        'pixel_auc': s.get('pixel_auc'),
        'pixel_precision': s.get('pixel_precision'),
        'pixel_recall': s.get('pixel_recall'),
        # --- image-level metrics ---
        'image_accuracy': s.get('image_accuracy'),
        'image_macro_f1': s.get('image_macro_f1'),
        'image_roc_auc': s.get('image_roc_auc'),
        # --- training ---
        'best_val_f1': s.get('val_pixel_f1'),
        'final_train_loss': s.get('train_loss'),
        'final_val_loss': s.get('val_loss'),
        'duration_min': s.get('_wandb', {}).get('runtime', 0) / 60,
    })

df = pd.DataFrame(records)
print(f'Collected {len(df)} run records')
print(f'Completed runs with pixel_f1: {df["pixel_f1"].notna().sum()}')
print(f'Columns: {list(df.columns)}')
df.head()

## Step 4 --- Pixel F1 Leaderboard (Top 10)

In [ ]:
leaderboard = (
    df[df['pixel_f1'].notna()]
    .sort_values('pixel_f1', ascending=False)
    .reset_index(drop=True)
)
leaderboard.index += 1
leaderboard.index.name = 'Rank'

lb_cols = ['version', 'change', 'feature_set', 'pixel_f1', 'pixel_iou',
           'pixel_auc', 'image_accuracy', 'run_tag', 'state']

print('=' * 90)
print('  PIXEL F1 LEADERBOARD --- Top 10')
print('=' * 90)
print()
print(tabulate(
    leaderboard[lb_cols].head(10),
    headers='keys', tablefmt='pipe', floatfmt='.4f', showindex=True
))
print(f'\nShowing top 10 of {len(leaderboard)} completed runs.')

### Image Accuracy Leaderboard

In [ ]:
img_lb = (
    df[df['image_accuracy'].notna()]
    .sort_values('image_accuracy', ascending=False)
    .reset_index(drop=True)
)
img_lb.index += 1
img_lb.index.name = 'Rank'

img_cols = ['version', 'change', 'image_accuracy', 'image_macro_f1',
            'image_roc_auc', 'pixel_f1', 'run_tag']

print('=' * 90)
print('  IMAGE ACCURACY LEADERBOARD --- Top 10')
print('=' * 90)
print()
print(tabulate(
    img_lb[img_cols].head(10),
    headers='keys', tablefmt='pipe', floatfmt='.4f', showindex=True
))

### Feature Set Comparison

In [ ]:
valid = df[df['pixel_f1'].notna()]
if len(valid) > 0 and valid['feature_set'].nunique() > 1:
    feat_summary = valid.groupby('feature_set').agg({
        'pixel_f1': ['mean', 'max', 'count'],
        'pixel_iou': ['mean', 'max'],
        'image_accuracy': ['mean', 'max'],
    }).round(4)
    feat_summary.columns = [
        'F1_mean', 'F1_max', 'runs',
        'IoU_mean', 'IoU_max',
        'ImgAcc_mean', 'ImgAcc_max'
    ]
    feat_summary = feat_summary.sort_values('F1_max', ascending=False)

    print('=' * 90)
    print('  FEATURE SET COMPARISON')
    print('=' * 90)
    print()
    print(tabulate(feat_summary, headers='keys', tablefmt='pipe',
                   floatfmt='.4f', showindex=True))
else:
    print('Not enough distinct feature sets for comparison.')

## Step 5 --- Export Results to CSV

In [ ]:
leaderboard.to_csv('leaderboard_results.csv')
print('Saved: leaderboard_results.csv')

df.to_csv('all_experiment_results.csv', index=False)
print('Saved: all_experiment_results.csv')

## Step 6 --- Performance Comparison Graphs

In [ ]:
plot_df = leaderboard.head(15).iloc[::-1].copy()  # reverse for horizontal bar

if len(plot_df) == 0:
    print('No completed runs to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, max(5, len(plot_df) * 0.4)))

    # --- Pixel F1 ---
    ax = axes[0]
    colors = sns.color_palette('viridis', n_colors=len(plot_df))
    ax.barh(plot_df['version'], plot_df['pixel_f1'], color=colors)
    ax.set_xlabel('Pixel F1')
    ax.set_title('Pixel F1 Comparison')
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    for i, v in enumerate(plot_df['pixel_f1']):
        if pd.notna(v):
            ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    # --- Pixel IoU ---
    ax = axes[1]
    ax.barh(plot_df['version'], plot_df['pixel_iou'], color=colors)
    ax.set_xlabel('Pixel IoU')
    ax.set_title('Pixel IoU Comparison')
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    for i, v in enumerate(plot_df['pixel_iou']):
        if pd.notna(v):
            ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    # --- Image Accuracy ---
    ax = axes[2]
    ax.barh(plot_df['version'], plot_df['image_accuracy'], color=colors)
    ax.set_xlabel('Image Accuracy')
    ax.set_title('Image Accuracy Comparison')
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    for i, v in enumerate(plot_df['image_accuracy']):
        if pd.notna(v):
            ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

    fig.suptitle('Performance Comparison --- Top 15 Experiments', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig('performance_plots.png', dpi=150, bbox_inches='tight')
    print('Saved: performance_plots.png')
    plt.show()

## Step 7 --- Ablation Study Plots (vR.P.x Progression)

In [ ]:
def extract_version_number(version_str):
    """Extract numeric sort key from version strings like 'vR.P.10' or 'vR.P.3.1'."""
    import re
    m = re.search(r'P\.(\d+(?:\.\d+)*)', str(version_str))
    if m:
        parts = m.group(1).split('.')
        return sum(int(p) / (100 ** i) for i, p in enumerate(parts))
    m = re.search(r'vR\.(\d+(?:\.\d+)*)', str(version_str))
    if m:
        parts = m.group(1).split('.')
        return sum(int(p) / (100 ** i) for i, p in enumerate(parts))
    return 999


ablation_df = df[df['pixel_f1'].notna()].copy()
ablation_df['sort_key'] = ablation_df['version'].apply(extract_version_number)
# Keep best run per version
ablation_df = (
    ablation_df.sort_values('pixel_f1', ascending=False)
    .drop_duplicates(subset='version', keep='first')
    .sort_values('sort_key')
    .reset_index(drop=True)
)

if len(ablation_df) < 2:
    print('Not enough experiments for ablation plot.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    # --- Pixel F1 progression ---
    ax = axes[0]
    ax.plot(ablation_df['version'], ablation_df['pixel_f1'],
            'o-', color='#2196F3', linewidth=2, markersize=6)
    ax.set_ylabel('Pixel F1')
    ax.set_title('Pixel F1 vs Experiment Version')
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
    # Annotate best
    best_idx = ablation_df['pixel_f1'].idxmax()
    best_row = ablation_df.loc[best_idx]
    ax.annotate(
        f'{best_row["pixel_f1"]:.4f}',
        xy=(best_row['version'], best_row['pixel_f1']),
        xytext=(0, 12), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold', color='#D32F2F',
        arrowprops=dict(arrowstyle='->', color='#D32F2F')
    )

    # --- IoU progression ---
    ax = axes[1]
    ax.plot(ablation_df['version'], ablation_df['pixel_iou'],
            's-', color='#4CAF50', linewidth=2, markersize=6)
    ax.set_ylabel('Pixel IoU')
    ax.set_xlabel('Experiment Version')
    ax.set_title('Pixel IoU vs Experiment Version')
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))

    plt.xticks(rotation=45, ha='right')
    fig.suptitle('Ablation Study --- Metric Progression Across vR.P.x Series',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig('ablation_summary.png', dpi=150, bbox_inches='tight')
    print('Saved: ablation_summary.png')
    plt.show()

## Step 8 --- Training Curve Visualization

In [ ]:
# Select top-5 runs by pixel_f1 to compare training curves
top_runs = leaderboard.head(5)

if len(top_runs) == 0:
    print('No completed runs for training curves.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    palette = sns.color_palette('tab10', n_colors=len(top_runs))

    for i, (_, row) in enumerate(top_runs.iterrows()):
        run = api.run(f'{entity}/{WANDB_PROJECT}/{row["wandb_run_id"]}')
        history = run.history(pandas=True)

        if history.empty:
            continue

        label = row['version']
        color = palette[i]

        # Train loss
        if 'train_loss' in history.columns:
            valid = history[history['train_loss'].notna()]
            axes[0].plot(valid['epoch'], valid['train_loss'],
                         label=label, color=color, linewidth=1.5)

        # Val loss
        if 'val_loss' in history.columns:
            valid = history[history['val_loss'].notna()]
            axes[1].plot(valid['epoch'], valid['val_loss'],
                         label=label, color=color, linewidth=1.5)

        # Val Pixel F1
        if 'val_pixel_f1' in history.columns:
            valid = history[history['val_pixel_f1'].notna()]
            axes[2].plot(valid['epoch'], valid['val_pixel_f1'],
                         label=label, color=color, linewidth=1.5)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Train Loss vs Epoch'); axes[0].legend(fontsize=8)

    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].set_title('Val Loss vs Epoch'); axes[1].legend(fontsize=8)

    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Pixel F1')
    axes[2].set_title('Val Pixel F1 vs Epoch'); axes[2].legend(fontsize=8)

    fig.suptitle('Training Curves --- Top 5 Experiments', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Step 9 --- Best Model Identification

In [ ]:
if len(leaderboard) > 0:
    best = leaderboard.iloc[0]

    print('=' * 70)
    print('  BEST EXPERIMENT')
    print('=' * 70)
    print(f'  Version:        {best["version"]}')
    print(f'  Change:         {best["change"]}')
    print(f'  Feature Set:    {best["feature_set"]}')
    print(f'  Encoder:        {best["encoder"]}')
    print(f'  Pixel F1:       {best["pixel_f1"]:.4f}')
    print(f'  Pixel IoU:      {best["pixel_iou"]:.4f}')
    print(f'  Pixel AUC:      {best["pixel_auc"]:.4f}' if pd.notna(best.get('pixel_auc')) else '')
    print(f'  Image Accuracy: {best["image_accuracy"]:.4f}' if pd.notna(best.get('image_accuracy')) else '')
    print(f'  Image Macro F1: {best["image_macro_f1"]:.4f}' if pd.notna(best.get('image_macro_f1')) else '')
    print(f'  Duration:       {best["duration_min"]:.1f} min')
    print(f'  W&B Run ID:     {best["wandb_run_id"]}')
    print(f'  Run URL:        https://wandb.ai/{entity}/{WANDB_PROJECT}/runs/{best["wandb_run_id"]}')
    print('=' * 70)
else:
    print('No completed runs found.')

## Step 10 --- Prediction Examples (from W&B Media)

In [ ]:
# Fetch prediction_examples images logged to W&B for top-3 runs
top3 = leaderboard.head(3)

for _, row in top3.iterrows():
    run = api.run(f'{entity}/{WANDB_PROJECT}/{row["wandb_run_id"]}')
    history = run.history(pandas=False)

    # Look for prediction_examples in logged media
    found = False
    for step in reversed(history):
        if 'prediction_examples' in step:
            img_data = step['prediction_examples']
            if isinstance(img_data, dict) and 'path' in img_data:
                img_url = img_data.get('path') or img_data.get('url', '')
                print(f'{row["version"]}: prediction_examples logged (media key found)')
                found = True
            break

    if not found:
        # Try fetching from run files
        media_files = [f for f in run.files() if 'prediction' in f.name.lower()]
        if media_files:
            print(f'{row["version"]}: {len(media_files)} prediction file(s) in run files')
            for mf in media_files[:3]:
                mf.download(replace=True)
                from PIL import Image as PILImage
                img = PILImage.open(mf.name)
                fig, ax = plt.subplots(1, 1, figsize=(12, 4))
                ax.imshow(img)
                ax.set_title(f'{row["version"]} --- Prediction Examples')
                ax.axis('off')
                plt.tight_layout()
                plt.show()
        else:
            print(f'{row["version"]}: no prediction examples found in W&B media')

## Step 11 --- Precision / Recall Scatter and Metric Heatmap

In [ ]:
metric_df = df[df['pixel_f1'].notna()].copy()

if len(metric_df) < 2:
    print('Not enough runs for analysis plots.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Precision vs Recall scatter ---
    ax = axes[0]
    has_pr = metric_df['pixel_precision'].notna() & metric_df['pixel_recall'].notna()
    pr_df = metric_df[has_pr]
    if len(pr_df) > 0:
        scatter = ax.scatter(
            pr_df['pixel_recall'], pr_df['pixel_precision'],
            c=pr_df['pixel_f1'], cmap='viridis', s=80, edgecolors='black', linewidth=0.5
        )
        plt.colorbar(scatter, ax=ax, label='Pixel F1')
        for _, r in pr_df.iterrows():
            ax.annotate(r['version'], (r['pixel_recall'], r['pixel_precision']),
                        fontsize=7, ha='center', va='bottom', alpha=0.8)
    ax.set_xlabel('Pixel Recall')
    ax.set_ylabel('Pixel Precision')
    ax.set_title('Precision vs Recall (colored by F1)')

    # --- Metric correlation heatmap ---
    ax = axes[1]
    hm_cols = ['pixel_f1', 'pixel_iou', 'pixel_auc', 'pixel_precision',
               'pixel_recall', 'image_accuracy', 'image_macro_f1']
    hm_data = metric_df[hm_cols].dropna(axis=1, how='all')
    if hm_data.shape[1] >= 2:
        corr = hm_data.corr()
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
                    vmin=-1, vmax=1, ax=ax, square=True)
        ax.set_title('Metric Correlation Heatmap')
    else:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
        ax.set_title('Metric Correlation Heatmap')

    plt.tight_layout()
    plt.show()

## Step 12 --- Summary Statistics

In [ ]:
valid = df[df['pixel_f1'].notna()]

if len(valid) == 0:
    print('No completed runs.')
else:
    print('=' * 70)
    print('  EXPERIMENT SUMMARY')
    print('=' * 70)
    print(f'  Total runs completed:    {len(valid)}')
    print(f'  Unique versions:         {valid["version"].nunique()}')
    print(f'  Total GPU time:          {valid["duration_min"].sum():.0f} min '
          f'({valid["duration_min"].sum() / 60:.1f} hours)')
    print()
    print(f'  --- Pixel F1 ---')
    print(f'  Best:    {valid["pixel_f1"].max():.4f}  '
          f'({valid.loc[valid["pixel_f1"].idxmax(), "version"]})')
    print(f'  Mean:    {valid["pixel_f1"].mean():.4f}')
    print(f'  Median:  {valid["pixel_f1"].median():.4f}')
    print(f'  Std:     {valid["pixel_f1"].std():.4f}')
    print()
    print(f'  --- Pixel IoU ---')
    print(f'  Best:    {valid["pixel_iou"].max():.4f}  '
          f'({valid.loc[valid["pixel_iou"].idxmax(), "version"]})')
    print(f'  Mean:    {valid["pixel_iou"].mean():.4f}')
    print()
    print(f'  --- Image Accuracy ---')
    ia = valid[valid['image_accuracy'].notna()]
    if len(ia) > 0:
        print(f'  Best:    {ia["image_accuracy"].max():.4f}  '
              f'({ia.loc[ia["image_accuracy"].idxmax(), "version"]})')
        print(f'  Mean:    {ia["image_accuracy"].mean():.4f}')
    else:
        print('  (no image accuracy data)')

    # Baseline comparison
    print()
    print(f'  --- Improvement Over Baseline ---')
    baseline_versions = ['vR.P.0', 'vR.P.1']
    baseline = valid[valid['version'].isin(baseline_versions)]
    if len(baseline) > 0:
        bl_f1 = baseline['pixel_f1'].max()
        best_f1 = valid['pixel_f1'].max()
        improvement = best_f1 - bl_f1
        print(f'  Baseline (best of {baseline_versions}): {bl_f1:.4f}')
        print(f'  Best overall:                         {best_f1:.4f}')
        print(f'  Absolute improvement:                 +{improvement:.4f} ({improvement/bl_f1*100:.1f}%)')
    else:
        print('  (baseline versions not found in data)')

    print('=' * 70)

## Step 13 --- Report Export and W&B Artifact Upload

In [ ]:
# --- Run Duration Summary ---
if 'duration_min' in df.columns:
    duration = df[['version', 'run_tag', 'duration_min', 'state']].sort_values(
        'duration_min', ascending=False)
    total_min = duration['duration_min'].sum()
    print(f'Total GPU time: {total_min:.0f} min ({total_min / 60:.1f} hours)')
    print(f'Average per run: {total_min / max(len(duration), 1):.0f} min')
    print()
    print(tabulate(duration, headers='keys', tablefmt='pipe',
                   floatfmt='.1f', showindex=False))

In [ ]:
# --- Upload leaderboard + plots as W&B artifact ---
import glob

export_files = [
    'leaderboard_results.csv',
    'all_experiment_results.csv',
    'performance_plots.png',
    'ablation_summary.png',
]

try:
    with wandb.init(
        project=WANDB_PROJECT,
        name='leaderboard_export',
        job_type='analysis',
        reinit=True,
    ):
        artifact = wandb.Artifact('leaderboard', type='results')
        for f in export_files:
            if os.path.exists(f):
                artifact.add_file(f)
                print(f'  Added to artifact: {f}')
        wandb.log_artifact(artifact)

        # Log leaderboard as W&B Table
        wandb.log({'leaderboard': wandb.Table(dataframe=leaderboard[lb_cols].head(20))})
        print('\nLeaderboard and artifacts uploaded to W&B.')
except Exception as e:
    print(f'W&B artifact upload skipped: {e}')

print('\n--- All exports complete ---')
for f in export_files:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')